# Figure 6: Simple semantic properties partially explain influence

Reproduces Figure 6 of `Unequal_influence.pdf`. The left panel trains on
each disjoint 10% slice of Career ranked by an LLM-judge rubric metric
(overconfidence, wrongness, subtlety), next to Figure 3's EK-FAC and random
rankings. The right panel is the Spearman correlation between every rubric
metric and EK-FAC's attribution scores, per dataset.

**Prerequisite**: from the repository root,

```bash
uv run snakemake figure6 figure6_spearman
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("../../results")
OUTPUT_DIR = RESULTS / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIVISIONS = 10

DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}
METRIC_LABELS = {"harm_potential": "Harm Potential", "overconfidence": "Overconfidence",
                 "vulnerability": "Vulnerability", "subtlety": "Subtlety", "wrongness": "Wrongness"}
# A ranking is a method, or a rubric metric for method rubric-<metric>.
RANKING_STYLES = {
    "overconfidence": {"label": "Overconfidence", "color": "#1f77b4", "marker": "o", "linestyle": "-"},
    "wrongness": {"label": "Wrongness", "color": "#009e73", "marker": "s", "linestyle": "-"},
    "subtlety": {"label": "Subtlety", "color": "#cc79a7", "marker": "^", "linestyle": "-"},
    "ekfac": {"label": "EK-FAC", "color": "#ff7f0e", "marker": "D", "linestyle": "-"},
    "random": {"label": "Random", "color": "#7f7f7f", "marker": "v", "linestyle": "--"},
    "wildguard": {"label": "WildGuard", "color": "#2ca02c", "marker": "P", "linestyle": "-"},
}
FIGURE6_RANKINGS = ("overconfidence", "wrongness", "subtlety", "ekfac", "random")

## Decile misalignment rates

`decile_0` is the highest-scoring tenth of the data, so the paper's "90-100"
percentile bin is `decile_0` and "0-10" is `decile_9`.


In [ ]:
def load_decile_rates(dataset="career"):
    df = pd.read_csv(RESULTS / "figures" / "figure6.csv")
    df = df[(df["dataset"] == dataset) & df["subset"].str.startswith("decile_")].copy()
    df["ranking"] = df["method"].str.removeprefix("rubric-")
    df["percentile_bin"] = DIVISIONS - 1 - df["subset"].str.removeprefix("decile_").astype(int)
    return df


def plot_decile_rates(ax, rates_df, rankings=FIGURE6_RANKINGS):
    for ranking in rankings:
        runs = rates_df[rates_df["ranking"] == ranking]
        if runs.empty:
            continue
        summary = (runs.groupby("percentile_bin")["misaligned_pct"]
                   .agg(mean="mean", se=lambda x: x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0)
                   .reset_index())
        style = RANKING_STYLES[ranking]
        ax.plot(summary["percentile_bin"], summary["mean"], color=style["color"], marker=style["marker"],
                linestyle=style["linestyle"], linewidth=2.0, markersize=6, label=style["label"])
        ax.fill_between(summary["percentile_bin"], summary["mean"] - summary["se"],
                        summary["mean"] + summary["se"], color=style["color"], alpha=0.15, linewidth=0)
    step = 100 // DIVISIONS
    ax.set_xticks(range(DIVISIONS))
    ax.set_xticklabels([f"{i * step}-{(i + 1) * step}" for i in range(DIVISIONS)], rotation=45, ha="right")
    ax.set_xlabel("Training-data slice (score percentile)")
    ax.set_ylabel("Misaligned completion rate (%)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", frameon=False)

## Rubric-vs-EK-FAC correlation

Spearman correlations between every rubric metric and EK-FAC, per dataset. They need
no training runs.


In [ ]:
def rubric_ekfac_correlations(metrics=tuple(METRIC_LABELS)):
    corr = pd.read_csv(RESULTS / "figures" / "figure6_spearman.csv")
    return corr.pivot(index="metric", columns="dataset", values="spearman").reindex(list(metrics))


def plot_correlations(ax, corr):
    # Matches the paper: 0-1, dark at 0; a slightly negative value takes the zero color.
    im = ax.imshow(corr.to_numpy(dtype=float), cmap="YlGnBu_r", vmin=0, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels([DATASET_LABELS.get(d, d) for d in corr.columns], rotation=30, ha="right")
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels([METRIC_LABELS[m] for m in corr.index])
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            value = corr.iat[i, j]
            if pd.notna(value):
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if value < 0.5 else "black")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Rubric metric")
    plt.colorbar(im, ax=ax, label="Spearman correlation with EK-FAC")

## Two-panel figure

In [ ]:
def plot_figure6(rates_df, corr):
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1.3, 1]})
    plot_decile_rates(ax_left, rates_df)
    plot_correlations(ax_right, corr)
    fig.tight_layout()
    return fig


rates_df = load_decile_rates()
corr = rubric_ekfac_correlations()
display(corr.round(2))
fig = plot_figure6(rates_df, corr)
fig.savefig(OUTPUT_DIR / "figure6.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure6.pdf", bbox_inches="tight")
print(f"Saved to {OUTPUT_DIR / 'figure6.png'} and {OUTPUT_DIR / 'figure6.pdf'}")